In [1]:
# Import necessary libraries
import pandas as pd
import re
from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import sys
sys.path.append("..")
from data.base import get_connection

# Initialize ArcGIS geocoder
geolocator = ArcGIS(timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=0.1)

In [2]:
# Query DuckDB to get every unique camera intersection grouped by year
query = """
    SELECT DISTINCT 
        DATE_PART('year', CAST(issue_date AS TIMESTAMP)) AS year,
        street_name,
        intersecting_street
    FROM tickets_all
    WHERE CAST(violation_code AS INTEGER) IN (7, 36)
      AND street_name IS NOT NULL 
      AND intersecting_street IS NOT NULL
"""

con = get_connection()
try:
    yearly_intersections = con.sql(query).df()
finally:
    con.close()

# Drop rows with invalid years
yearly_intersections = yearly_intersections.dropna(subset=['year'])
yearly_intersections['year'] = yearly_intersections['year'].astype(int)

print(f"Total yearly intersection records found: {len(yearly_intersections)}")
yearly_intersections.head()

Total yearly intersection records found: 22062


,year,street_name,intersecting_street
0,2017,SB BOSTON RD @ DE RE,IMER AVE
1,2017,EB 65TH ST @ 16TH AV,E
2,2017,SB CROPSEY AVE @ BAY,35TH ST
3,2017,EASTERN PKWY (W/B) @,BEDFORD AVE
4,2016,QUEENS BLVD (E/B) @,JACOBUS ST


In [3]:
# Combine the two street columns, filling missing values with blanks
raw_text = yearly_intersections['street_name'].fillna('') + ' ' + yearly_intersections['intersecting_street'].fillna('')

# Apply your exact FY2024 regex cleaning pipeline, adding the NYC tag for ArcGIS
yearly_intersections['search_query'] = (
    raw_text
    .str.upper()
    .str.replace(r'\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\b', '', regex=True)
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.replace('@', ' AND ')
    .str.replace(r'^\s+', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    + ', NEW YORK, NY'
)

yearly_intersections.head()

,year,street_name,intersecting_street,search_query
0,2017,SB BOSTON RD @ DE RE,IMER AVE,"BOSTON RD AND DE RE IMER AVE, NEW YORK, NY"
1,2017,EB 65TH ST @ 16TH AV,E,"65TH ST AND 16TH AV E, NEW YORK, NY"
2,2017,SB CROPSEY AVE @ BAY,35TH ST,"CROPSEY AVE AND BAY 35TH ST, NEW YORK, NY"
3,2017,EASTERN PKWY (W/B) @,BEDFORD AVE,"EASTERN PKWY AND BEDFORD AVE, NEW YORK, NY"
4,2016,QUEENS BLVD (E/B) @,JACOBUS ST,"QUEENS BLVD AND JACOBUS ST, NEW YORK, NY"


In [4]:
# Extract only the globally unique intersection strings across all 2016-2025 data
unique_queries = pd.DataFrame(yearly_intersections['search_query'].unique(), columns=['search_query'])

print(f"Total globally unique intersections to geocode: {len(unique_queries)}")

Total globally unique intersections to geocode: 4529


In [6]:
from tqdm import tqdm
# Explicitly enable progress_apply for Pandas
tqdm.pandas(desc="Geocoding Intersections")

# Run the geocoder against the rate limiter
unique_queries['Location_Data'] = unique_queries['search_query'].progress_apply(geocode)

# Extract precise latitude and longitude
unique_queries['Latitude'] = unique_queries['Location_Data'].apply(lambda loc: loc.latitude if loc else None)
unique_queries['Longitude'] = unique_queries['Location_Data'].apply(lambda loc: loc.longitude if loc else None)

# Drop the raw location object to save memory
unique_queries = unique_queries.drop(columns=['Location_Data']).dropna(subset=['Latitude', 'Longitude'])

Geocoding Intersections: 100%|██████████| 4529/4529 [34:03<00:00,  2.22it/s]


In [9]:
# Merge the geocoded coordinates back onto the yearly breakdown
yearly_cameras = yearly_intersections.merge(unique_queries, on='search_query', how='left')

# Save the final time-series spatial map
output_path = "../data/yearly_camera_coordinates.csv"
yearly_cameras.to_csv(output_path, index=False)

print(f"Success! Yearly camera locations saved to {output_path}")
yearly_cameras.sample(5)

Success! Yearly camera locations saved to ../data/yearly_camera_coordinates.csv


,year,street_name,intersecting_street,search_query,Latitude,Longitude
19725,2024,WB W 230TH ST @ TIBB,ET AVE,"W 230TH ST AND TIBB ET AVE, NEW YORK, NY",40.879514,-73.909689
13718,2025,EB 58TH AVE @ 185TH,ST,"58TH AVE AND 185TH ST, NEW YORK, NY",40.741865,-73.790190
13696,2021,SB 16 AV. DAHILL RD,- 45 ST,"16 AV. DAHILL RD - 45 ST, NEW YORK, NY",40.646792,-73.981331
4626,2025,WB N CONDUIT AVE @ 1,27TH ST,"N CONDUIT AVE AND 1 27TH ST, NEW YORK, NY",40.667027,-73.813276
5939,2024,BEACH CHANNEL DR (S/,B) @ HASSOCK ST,"BEACH CHANNEL DR AND HASSOCK ST, NEW YORK, NY",40.609514,-73.753719


In [2]:
import sys
sys.path.append("..")
import pandas as pd
from data.base import get_connection

# 1. Query DuckDB to retrieve year, violation_code, and intersection fields
query = """
    SELECT DISTINCT 
        DATE_PART('year', CAST(issue_date AS TIMESTAMP)) AS year,
        CAST(violation_code AS INTEGER) AS violation_code,
        street_name,
        intersecting_street
    FROM tickets_all
    WHERE CAST(violation_code AS INTEGER) IN (7, 36)
      AND street_name IS NOT NULL 
      AND intersecting_street IS NOT NULL
"""

con = get_connection()
try:
    yearly_typed = con.sql(query).df()
finally:
    con.close()

yearly_typed = yearly_typed.dropna(subset=['year'])
yearly_typed['year'] = yearly_typed['year'].astype(int)

# 2. Standardize search queries matching your existing regex pipeline
raw_text = yearly_typed['street_name'].fillna('') + ' ' + yearly_typed['intersecting_street'].fillna('')
yearly_typed['search_query'] = (
    raw_text
    .str.upper()
    .str.replace(r'\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\b', '', regex=True)
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.replace('@', ' AND ')
    .str.replace(r'^\s+', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    + ', NEW YORK, NY'
)

# 3. Label camera types
yearly_typed['camera_type'] = yearly_typed['violation_code'].map({
    7: 'Red Light',
    36: 'Speed'
})

# 4. Load the 4,529 geocoded coordinates produced in Cell 6 of this notebook
coords = pd.read_csv('../data/yearly_camera_coordinates.csv')[
    ['search_query', 'Latitude', 'Longitude']
].drop_duplicates(subset=['search_query'])

yearly_cameras_typed = yearly_typed.merge(coords, on='search_query', how='inner')

# 5. Overwrite the master file and export individual subsets
yearly_cameras_typed.to_csv('../data/yearly_camera_coordinates.csv', index=False)
yearly_cameras_typed[yearly_cameras_typed['camera_type'] == 'Red Light'].to_csv(
    '../data/yearly_red_light_cameras.csv', index=False
)
yearly_cameras_typed[yearly_cameras_typed['camera_type'] == 'Speed'].to_csv(
    '../data/yearly_speed_cameras.csv', index=False
)

print(f"Master file updated: {len(yearly_cameras_typed):,} rows")
print(f"  - Red Light camera records: {len(yearly_cameras_typed[yearly_cameras_typed['camera_type'] == 'Red Light']):,}")
print(f"  - Speed camera records:     {len(yearly_cameras_typed[yearly_cameras_typed['camera_type'] == 'Speed']):,}")

Master file updated: 22,133 rows
  - Red Light camera records: 2,992
  - Speed camera records:     19,141


In [9]:
import pandas as pd
import sys
sys.path.append("..")
from data.base import get_connection

# 1. Query DuckDB to retrieve year, violation_code, and intersection fields
query = """
    SELECT DISTINCT 
        DATE_PART('year', CAST(issue_date AS TIMESTAMP)) AS year,
        CAST(violation_code AS INTEGER) AS violation_code,
        street_name,
        intersecting_street
    FROM tickets_all
    WHERE CAST(violation_code AS INTEGER) IN (7, 36)
      AND street_name IS NOT NULL 
      AND intersecting_street IS NOT NULL
"""

con = get_connection()
try:
    yearly_typed = con.sql(query).df()
finally:
    con.close()

yearly_typed = yearly_typed.dropna(subset=['year'])
yearly_typed['year'] = yearly_typed['year'].astype(int)

# 2. Standardize search queries
raw_text = yearly_typed['street_name'].fillna('') + ' ' + yearly_typed['intersecting_street'].fillna('')
yearly_typed['search_query'] = (
    raw_text
    .str.upper()
    .str.replace(r'\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\b', '', regex=True)
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.replace('@', ' AND ')
    .str.replace(r'^\s+', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    + ', NEW YORK, NY'
)

# 3. Label camera types
yearly_typed['camera_type'] = yearly_typed['violation_code'].map({
    7: 'Red Light',
    36: 'Speed'
})

# 4. Load the existing geocoded coordinates
coords = pd.read_csv('../data/yearly_camera_coordinates.csv')[
    ['search_query', 'Latitude', 'Longitude']
].drop_duplicates(subset=['search_query'])

yearly_cameras_typed = yearly_typed.merge(coords, on='search_query', how='inner')

# 5. Overwrite the master CSVs
yearly_cameras_typed.to_csv('../data/yearly_camera_coordinates.csv', index=False)
yearly_cameras_typed[yearly_cameras_typed['camera_type'] == 'Red Light'].to_csv(
    '../data/yearly_red_light_cameras.csv', index=False
)
yearly_cameras_typed[yearly_cameras_typed['camera_type'] == 'Speed'].to_csv(
    '../data/yearly_speed_cameras.csv', index=False
)

# 6. Export multi-sheet Excel workbook
excel_path = '../data/yearly_camera_coordinates.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Master summary tab
    yearly_cameras_typed.to_excel(writer, sheet_name='All_Years', index=False)
    
    # Individual tab per year
    for yr in sorted(yearly_cameras_typed['year'].unique()):
        subset = yearly_cameras_typed[yearly_cameras_typed['year'] == yr]
        subset.to_excel(writer, sheet_name=str(yr), index=False)

print(f"Successfully generated:")
print(f"  - Master CSV: ../data/yearly_camera_coordinates.csv")
print(f"  - Red Light CSV: ../data/yearly_red_light_cameras.csv")
print(f"  - Speed CSV: ../data/yearly_speed_cameras.csv")
print(f"  - Multi-Sheet Excel: {excel_path}")

Successfully generated:
  - Master CSV: ../data/yearly_camera_coordinates.csv
  - Red Light CSV: ../data/yearly_red_light_cameras.csv
  - Speed CSV: ../data/yearly_speed_cameras.csv
  - Multi-Sheet Excel: ../data/yearly_camera_coordinates.xlsx
